# Descriptors from machine learning potentials

Machine learnings potentials (MLPs) are neural networks that are trained to predict energies from 3D coordinates. At the same time, they also learn an internal representation of each atom in the molecule, that can be used for other machine learning applications. One approach is to extract these representations and use them instead of traditional descriptors (such as fingerprints or quantum-chemical descriptors).

In another development, modern MLPs (such as MACE-Polar and AIMNet2) also treat electrostatics through explicit modeling of atomic charges and other multipoles. These can also be extracted, although it is not yet clear how useful they are in place of standard charges or spin densities. 

In this notebook, we will look at:

1. Whole-molecule descriptors generated by MACE-MP-0
1. Spin densities from the MACE-Polar-1-S model
1. Charge densities from the MACE-Polar-1-S model

## Set up

We will import the needed packages and silence some warnings that will otherwise clutter the output. We also wrote some convenience functions in `utils.py` that will make your life easier. You can have a look in that file if you want to understand more in depth, but it is not central for this exercise.

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.stats
from ase.io import read
from IPython.display import Image, display
from mace.calculators import mace_mp, mace_polar
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm import tqdm

import utils
from utils import rdkit_mol_to_ase_atoms

## MACE descriptors 

We will now calculate the so-called MACE descriptors, an internal representation of the MACE models. This is currently available only for some of the MACE models, so here we will give an example for the MACE-MP-0 model.

In [ ]:
calc = mace_mp(model="small", default_dtype="float64", device="cpu")

The descriptors are obtained with the function `get_descriptors` and have the shape `(n_atoms, n_descriptors)`. That means that the MACE descriptors can be used as atomic descriptors. We can take a look at one of the vectors just to get a feeling for how it looks like.

In [ ]:
nitrobenzene = read("data/descriptors/nitrobenzene.xyz")
descriptors = calc.get_descriptors(nitrobenzene)
print(f"Shape of descriptor array: {descriptors.shape}")
descriptors[0]

 If we want to have a whole-molecule descriptors, we can take the mean value of the descriptors across the atoms.

In [ ]:
molecule_descriptors = np.mean(descriptors, axis=0)
print(f"Shape of molecule descriptor: {molecule_descriptors.shape}")

## Computing descriptors for the ESOL dataset

We will now compute MACE descriptors for the [ESOL](https://doi.org/10.1021/ci034243x) dataset and do some regression modeling to predict the aqeuous solubility of these compounds. We will compare with standard Morgan fingerprints. To avoid having to do hyperparameter tuning, we will use a Random Forest model with standard parameters, which usually is very insensitive to tuning.

To calculate MACE descriptors, we need 3D geometries. These could be obtained by some generative 3D model followed by optimization with MLPs, but here we will make it easy for ourselves and just generate one conformer with RDKit and optimize with the MMFF force field.

In [ ]:
esol_df = pd.read_csv("data/descriptors/esol.csv")

We will now generate the 3D geometries and relax them with the MMFF force field

In [ ]:
tqdm.pandas(desc="Processing molecules")
mols = esol_df["smiles"].apply(Chem.MolFromSmiles)
mols = mols.apply(Chem.AddHs)
_ = mols.progress_apply(AllChem.EmbedMolecule)
_ = mols.progress_apply(AllChem.MMFFOptimizeMolecule)

Then we are ready to calculate the MACE descriptors. To do that, we first need to convert the RDKit Mol objects into the ASE Atoms objects that the MACE calculator expects. Then we calculate the descriptors. This will take a few minutes to calculate on the CPU.

In [ ]:
ase_atoms = mols.apply(rdkit_mol_to_ase_atoms)
mace_descriptors = ase_atoms.progress_apply(
    lambda x: calc.get_descriptors(x).mean(axis=0)
)

We go ahead and save the descriptors to disk so that we don't need to compute them again if needed.

In [ ]:
mace_descriptor_df = pd.DataFrame(mace_descriptors.tolist())

ESOL_OUTPUT_PATH = Path("output/descriptors/esol_mace_mp_small_mmff.csv")
ESOL_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
mace_descriptor_df.to_csv(ESOL_OUTPUT_PATH, index=False)
mace_descriptor_df.head()

Now we are ready to do some machine learning with our new descriptors! We will use a simple Random Forest model and evaluate it using 10-fold cross-validation to get some regression metrics and associated standard errors of the mean. You might get some UserWarnings that can safely be ignored.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_validate

X = mace_descriptor_df
y = esol_df["measured log solubility in mols per litre"]

cv = KFold(n_splits=10, shuffle=True, random_state=42)
metrics = ["r2", "mae", "rmse"]
scoring = ["r2", "neg_mean_absolute_error", "neg_root_mean_squared_error"]

cv_results = cross_validate(
    RandomForestRegressor(random_state=42),
    X,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
)

The cross-validation is done. Time to compute some regression metric means and standard errors.

In [ ]:
mace_scores = {
    "r2": cv_results["test_r2"],
    "mae": -cv_results["test_neg_mean_absolute_error"],
    "rmse": -cv_results["test_neg_root_mean_squared_error"],
}
mace_means = {metric: np.mean(scores) for metric, scores in mace_scores.items()}
mace_sems = {metric: scipy.stats.sem(scores) for metric, scores in mace_scores.items()}

for metric in metrics:
    print(f"{metric}: {mace_means[metric]:.4f} ± {mace_sems[metric]:.4f}")

How good are those results? Let's compare against a baseline, Morgan fingerprints.

In [ ]:
# Calculate the Morgan fingerprints
generator = AllChem.GetMorganGenerator(radius=2, fpSize=2048)
fps = mols.apply(generator.GetFingerprintAsNumPy)
X_morgan = pd.DataFrame(fps.tolist())

# Do the cross-validation
cv_results_morgan = cross_validate(
    RandomForestRegressor(random_state=42),
    X_morgan,
    y,
    cv=cv,
    scoring=scoring,
    n_jobs=-1,
)

# Calculate the results
morgan_scores = {
    "r2": cv_results_morgan["test_r2"],
    "mae": -cv_results_morgan["test_neg_mean_absolute_error"],
    "rmse": -cv_results_morgan["test_neg_root_mean_squared_error"],
}
morgan_means = {metric: np.mean(scores) for metric, scores in morgan_scores.items()}
morgan_sems = {
    metric: scipy.stats.sem(scores) for metric, scores in morgan_scores.items()
}

for metric in metrics:
    print(f"{metric}: {morgan_means[metric]:.4f} ± {morgan_sems[metric]:.4f}")

Let's look at that graphically with a bar chart and associated error bars corresponding to ± 1 standard error

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, metric in zip(axes, ["rmse", "mae", "r2"]):
    ax.bar(
        ["MACE MP Small", "Morgan FP"],
        [mace_means[metric], morgan_means[metric]],
        yerr=[mace_sems[metric], morgan_sems[metric]],
        capsize=5,
        ecolor="C1",
    )
    ax.set_title(metric.upper())

> How does the MACE descriptors hold up against Morgan fingerprints on the ESOL dataset? Are the differences significant?

> How could we build even better property prediction models based on foundational MLPs? 

> Compare your results to those listed on the [MoleculeNet](https://moleculenet.org/full-results) webpage. How are you doing?

## Spin densities

Newer models like MACE-Polar make use of multipoles for calculation of electrostatics. You can extract for example "atomic charges" and "spin densities" from these models. Currently, it is unclear how these would correspond to the classical atomic charges and spins that you can get from quantum-chemical calculations (and which particular population analysis they would compare to).

Let's calculate the triplet state of anthracene and look at some spin densities.

In [ ]:
calc_polar = mace_polar(model="polar-1-s", default_dtype="float64", device="cpu")

ANTHRACENE_XYZ_PATH = "data/descriptors/anthracene.xyz"
anthracene_triplet = read(ANTHRACENE_XYZ_PATH)
anthracene_triplet.info.update({"charge": 0, "spin": 3})
anthracene_triplet.calc = calc_polar
_ = (
    anthracene_triplet.get_potential_energy()
)  # We need to calculate the energy to populate the density_coefficients results.

We can now extract the spin densities. MACE-Polar splits the spin density into two channels: (1) spin-up charges (⍺ spin) and (2) spin-down charges (β spin). For a triplet, the value of the spin-down charges are negative, which corresponds to positive ⍺ spin.

In [ ]:
spin_charge_densities = calc_polar.results["spin_charge_density"]
charges_up = spin_charge_densities[:, 0, 0]
charges_down = spin_charge_densities[:, 1, 0]
spin_densities = spin_charge_densities[:, 0, 0] - spin_charge_densities[:, 1, 0]

print(f"Charges up sum: {charges_up.sum()}")
print(f"Charges down sum: {charges_down.sum()}")
print(f"Spin density sum: {spin_densities.sum()}")

Now that we have extracted the spin densities from the model, we can visualize it (using RDKit) and compare to the corresponding values for the DFT level on which the MACE-Polar model was trained on.

In [ ]:
# Load the reference spin densities from DFT
df_spin_densities = pd.read_csv("data/descriptors/anthracene_spin_densities.csv")
spin_densities_dft = df_spin_densities["Spin density"].values

# Plot the spin densities
print("MACE-Polar spin density map:")
display(
    Image(
        utils.draw_similarity_map_png_from_xyz(
            ANTHRACENE_XYZ_PATH,
            spin_densities,
        )
    )
)

print("DFT spin density map:")
display(
    Image(
        utils.draw_similarity_map_png_from_xyz(
            ANTHRACENE_XYZ_PATH,
            spin_densities_dft,
        )
    )
)

# Calculate atom-wise differences
delta = spin_densities - spin_densities_dft
display_df = pd.DataFrame(
    {"MACE_spin": spin_densities, "DFT_spin": spin_densities_dft, "Delta": delta}
)
display(display_df)
print(f"MAE vs DFT spin:  {display_df['Delta'].abs().mean():.3f}")
print(f"RMSE vs DFT spin: {np.sqrt(display_df['Delta'].pow(2).mean()):.3f}")

> How do the "spin density" from MACE-Polar compare to DFT? What are the similarities and differences?

## Charge densities

Now we will calculate the charge densities from MACE, correspond to a type of atomic charges. We will compare them with the DFT-calculated values for the Mulliken (highly basis set-dependent) and the Löwdin charges (not so basis set-dependent). 

In [ ]:
NITROBENZENE_XYZ_PATH = "data/descriptors/nitrobenzene.xyz"
nitrobenzene = read(NITROBENZENE_XYZ_PATH)
nitrobenzene.info.update({"charge": 0, "spin": 1})
nitrobenzene.calc = calc_polar
_ = nitrobenzene.get_potential_energy()  # populate results

partial_charges = calc_polar.results["density_coefficients"][:, 0]
print("Partial charges from MACE-Polar:")
for i, charge in enumerate(partial_charges):
    print(f"Atom {i}: {charge:+.2f}")

We can now compare them visually to the ones from DFT.

In [ ]:
df_charges = pd.read_csv("data/descriptors/nitrobenzene_charges.csv")
charges_dft_mulliken = df_charges["Mulliken Charge"].values
charges_dft_lowdin = df_charges["Löwdin Charge"].values

print("MACE-Polar charge map:")
display(
    Image(
        utils.draw_similarity_map_png_from_xyz(
            NITROBENZENE_XYZ_PATH,
            partial_charges,
        )
    )
)

print("DFT Mulliken charge map:")
display(
    Image(
        utils.draw_similarity_map_png_from_xyz(
            NITROBENZENE_XYZ_PATH,
            charges_dft_mulliken,
        )
    )
)

print("DFT Löwdin charge map:")
display(
    Image(
        utils.draw_similarity_map_png_from_xyz(
            NITROBENZENE_XYZ_PATH,
            charges_dft_lowdin,
        )
    )
)

> How do the charges compare to each other? What is the "ground truth"?

## Bonus tasks

If you are out of things to do:

- How are the charges and spin densities affected by using the larger MACE-Polar-1-M and MACE-Polar-1-L models? 
- How does the chemical space of the molecules in the ESOL dataset look like based on MACE descriptors? Try to make a PCA, UMAP or t-SNE.